# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Signal A check: staleness vs. decline

In [2]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)

bucket_a = df.groupby("stale").agg(n=("content_id", "count"), decline_rate=("is_declining", "mean"))
bucket_a

,n,decline_rate
stale,,
0,29826,0.542480
1,174,0.471264


### Signal B check: CTR vs. position tier

In [3]:
bucket_b = df.groupby("position_tier").agg(n=("content_id", "count"), mean_ctr=("ctr", "mean")).sort_values("mean_ctr", ascending=False)
bucket_b

,n,mean_ctr
position_tier,,
top_3,2321,1.483611
page_1,11814,0.652467
striking,7304,0.323239
page_3_5,7242,0.222484
deep,1319,0.150212


##### A page is worth reviewing for a CTR fix if it's genuinely visible, it holds a good average position, and yet its CTR is low relative to what that position should earn. That combination -> good position, low CTR -> the signal that something about the page itself is costing clicks it should be getting for free.

##### One reason code: "visible_but_low_ctr_at_good_position"
##### Action label: "review_for_ctr_fix" else "no_action"

##### I dropped the staleness signal from the score itself. Once I saw Signal A's verdict, I didn't want to build a rule on a signal that didn't hold up.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
visible = (df["impressions_90d"] >= 500).astype(int)
good_position = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
low_ctr = (df["ctr"] < 0.3).astype(int)

df["score"] = visible * good_position.astype(int) * low_ctr * df["impressions_90d"]
df["reason_code"] = "visible_but_low_ctr_at_good_position"
df["action"] = np.where(df["score"] > 0, "review_for_ctr_fix", "no_action")

queue = df.sort_values("score", ascending=False)
qualifying = (df["score"] > 0).sum()
print(f"Qualifying rows: {qualifying} ({qualifying/len(df):.1%} of dataset)")

Qualifying rows: 7555 (25.2% of dataset)


In [5]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written to work/outputs/baseline_action_score.csv")
queue.head(10)[["content_id", "client_id", "impressions_90d", "ctr", "avg_position", "reason_code", "action", "score"]]

Written to work/outputs/baseline_action_score.csv


,content_id,client_id,impressions_90d,ctr,avg_position,reason_code,action,score
6653,content_5fe46e04994d,client_4e07408562,517715,0.14,4.2,visible_but_low_ctr_at_good_position,review_for_ctr_fix,517715
17812,content_aaef01a50def,client_19581e27de,517109,0.25,5.4,visible_but_low_ctr_at_good_position,review_for_ctr_fix,517109
26844,content_8c19996aa890,client_4e07408562,509252,0.15,2.5,visible_but_low_ctr_at_good_position,review_for_ctr_fix,509252
29879,content_1a9e894be2e2,client_19581e27de,416180,0.23,4.0,visible_but_low_ctr_at_good_position,review_for_ctr_fix,416180
18870,content_db5989a78dd3,client_4e07408562,345111,0.21,5.4,visible_but_low_ctr_at_good_position,review_for_ctr_fix,345111
26531,content_cb112fce36be,client_19581e27de,309910,0.16,5.6,visible_but_low_ctr_at_good_position,review_for_ctr_fix,309910
3394,content_36ff89c8214e,client_19581e27de,295097,0.05,7.3,visible_but_low_ctr_at_good_position,review_for_ctr_fix,295097
7678,content_8451fc6f034d,client_d029fa3a95,272144,0.03,2.3,visible_but_low_ctr_at_good_position,review_for_ctr_fix,272144
27478,content_008fb02c46cb,client_349c41201b,236803,0.26,4.4,visible_but_low_ctr_at_good_position,review_for_ctr_fix,236803
6903,content_c84a0ab98e90,client_f369cb89fc,223271,0.03,7.8,visible_but_low_ctr_at_good_position,review_for_ctr_fix,223271


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20)[["content_id", "client_id", "impressions_90d", "ctr", "avg_position", "days_since_last_update", "reason_code", "action", "score"]]
top20

,content_id,client_id,impressions_90d,ctr,avg_position,days_since_last_update,reason_code,action,score
6653,content_5fe46e04994d,client_4e07408562,517715,0.14,4.2,104,visible_but_low_ctr_at_good_position,review_for_ctr_fix,517715
17812,content_aaef01a50def,client_19581e27de,517109,0.25,5.4,22,visible_but_low_ctr_at_good_position,review_for_ctr_fix,517109
26844,content_8c19996aa890,client_4e07408562,509252,0.15,2.5,20,visible_but_low_ctr_at_good_position,review_for_ctr_fix,509252
29879,content_1a9e894be2e2,client_19581e27de,416180,0.23,4.0,22,visible_but_low_ctr_at_good_position,review_for_ctr_fix,416180
18870,content_db5989a78dd3,client_4e07408562,345111,0.21,5.4,20,visible_but_low_ctr_at_good_position,review_for_ctr_fix,345111
26531,content_cb112fce36be,client_19581e27de,309910,0.16,5.6,104,visible_but_low_ctr_at_good_position,review_for_ctr_fix,309910
3394,content_36ff89c8214e,client_19581e27de,295097,0.05,7.3,104,visible_but_low_ctr_at_good_position,review_for_ctr_fix,295097
7678,content_8451fc6f034d,client_d029fa3a95,272144,0.03,2.3,20,visible_but_low_ctr_at_good_position,review_for_ctr_fix,272144
27478,content_008fb02c46cb,client_349c41201b,236803,0.26,4.4,20,visible_but_low_ctr_at_good_position,review_for_ctr_fix,236803
6903,content_c84a0ab98e90,client_f369cb89fc,223271,0.03,7.8,20,visible_but_low_ctr_at_good_position,review_for_ctr_fix,223271


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
score_inputs = ["impressions_90d", "avg_position", "ctr"]
leak_cols = ["trend_direction", "trend_pct"]

future_ish_cols = [c for c in df.columns if "last_30d" in c or "prev_30d" in c]
future_ish_cols

['impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d']

##### - Whether any top-10 row has a suspiciously extreme CTR (near 0) that might be a data artifact rather than a real snippet problem.
##### - Whether one client dominates the top of the queue (check `client_id` in the top 10 -- if it's the same client repeatedly, that's a concentration risk worth naming, not hiding).
##### - Confirm: the rule uses only `impressions_90d`, `avg_position`, and `ctr` -- all observed-to-date signals, no `trend_direction`/`trend_pct`, no `*_last_30d` future-adjacent columns.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.